# Cohort tumor and PerturbView decoding QC

Read an exported `cohort_cell_analysis.h5ad` without loading its intensity matrix into memory. This notebook summarizes persisted per-cell decoding fields, visualizes tumor assignments and called guides in slide-local micron coordinates, and provides a selected-slide zoom. It does not rerun decoding or modify the H5AD.

Tumor polygons are not embedded in the cohort H5AD, so tumor maps show the spatial cloud of cells assigned to each tumor rather than the original GeoJSON boundary.

In [ ]:
from pathlib import Path
import re

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300})

## Inputs

Backed mode leaves the large cell-by-channel intensity matrix (`X` and layers) on disk. Observation annotations and spatial coordinates are loaded because they are the data being inspected.

In [ ]:
COHORT_H5AD = Path("/path/to/post_analysis/cohort_cell_analysis.h5ad")
NO_CALL_LABEL = "None"
UNKNOWN_LABEL = "UNK"
UNASSIGNED_TUMOR = "unassigned"
MAX_POINTS_PER_SLIDE = 300_000
POINT_SIZE = 0.25
RANDOM_SEED = 0

cohort = ad.read_h5ad(COHORT_H5AD, backed="r")
obs = cohort.obs
spatial = np.asarray(cohort.obsm["spatial"], dtype=np.float64)

required = {"slide_id", "tumor_id", "decode_eligible", "decode_incomplete_input", "decode_guide_call"}
missing = sorted(required.difference(obs.columns))
if missing:
    raise KeyError(f"Cohort H5AD is missing required observation fields: {missing}")
if spatial.shape != (cohort.n_obs, 2):
    raise ValueError(f"Expected obsm['spatial'] shape {(cohort.n_obs, 2)}, got {spatial.shape}")

round_names = []
for column in obs.columns:
    match = re.fullmatch(r"decode_(.+)_pass", str(column))
    if match:
        round_names.append(match.group(1))
if not round_names:
    raise KeyError("No per-round decode_*_pass fields were found.")

slide_values = obs["slide_id"].astype(str).to_numpy()
tumor_values = obs["tumor_id"].astype(str).to_numpy()
guide_values = obs["decode_guide_call"].astype(str).to_numpy()
eligible = obs["decode_eligible"].fillna(False).to_numpy(dtype=bool)
incomplete = obs["decode_incomplete_input"].fillna(True).to_numpy(dtype=bool)
tumor_assigned = tumor_values != UNASSIGNED_TUMOR
any_call = eligible & (guide_values != NO_CALL_LABEL)
mapped_call = any_call & (guide_values != UNKNOWN_LABEL)
unknown_call = any_call & (guide_values == UNKNOWN_LABEL)
all_rounds_pass = eligible & np.logical_and.reduce([
    obs[f"decode_{name}_pass"].fillna(False).to_numpy(dtype=bool)
    for name in round_names
])

print(cohort)
print(f"Slides: {pd.unique(slide_values).tolist()}")
print(f"Detected decoding rounds: {round_names}")

## Compact decoding metrics

The primary denominator is `eligible_nuclear_cells`: tumor-assigned cells with complete finite nuclear decoding measurements. `any_call` includes both mapped guides and `UNK`; `mapped_guide` excludes `None` and `UNK`.

In [ ]:
flags = pd.DataFrame({
    "slide_id": slide_values,
    "total_cells": True,
    "tumor_assigned": tumor_assigned,
    "complete_nuclear_input": ~incomplete,
    "eligible_nuclear_cells": eligible,
    "all_rounds_pass": all_rounds_pass,
    "any_call": any_call,
    "mapped_guide": mapped_call,
    "unknown_tuple": unknown_call,
})
counts = flags.groupby("slide_id", sort=False).sum().astype(np.int64)
counts.loc["COHORT"] = flags.drop(columns="slide_id").sum().astype(np.int64)
counts["fraction_eligible_among_tumor"] = counts["eligible_nuclear_cells"] / counts["tumor_assigned"]
counts["fraction_any_call_among_eligible"] = counts["any_call"] / counts["eligible_nuclear_cells"]
counts["fraction_mapped_guide_among_eligible"] = counts["mapped_guide"] / counts["eligible_nuclear_cells"]
counts["fraction_unknown_among_eligible"] = counts["unknown_tuple"] / counts["eligible_nuclear_cells"]
counts["fraction_all_rounds_pass_among_eligible"] = counts["all_rounds_pass"] / counts["eligible_nuclear_cells"]
display(counts)

In [ ]:
round_rows = []
eligible_index = np.flatnonzero(eligible)
for round_name in round_names:
    frame = pd.DataFrame({
        "slide_id": slide_values[eligible_index],
        "pass_raw_threshold": obs[f"decode_{round_name}_pass_top"].iloc[eligible_index].to_numpy(dtype=bool),
        "pass_ratio": obs[f"decode_{round_name}_pass_ratio"].iloc[eligible_index].to_numpy(dtype=bool),
        "pass_both": obs[f"decode_{round_name}_pass"].iloc[eligible_index].to_numpy(dtype=bool),
        "ratio": obs[f"decode_{round_name}_ratio"].iloc[eligible_index].to_numpy(dtype=float),
        "top_fold": obs[f"decode_{round_name}_top_fold"].iloc[eligible_index].to_numpy(dtype=float),
    })
    grouped = frame.groupby("slide_id", sort=False).agg(
        n_eligible=("pass_both", "size"),
        pass_raw_threshold=("pass_raw_threshold", "mean"),
        pass_ratio=("pass_ratio", "mean"),
        pass_both=("pass_both", "mean"),
        median_ratio=("ratio", "median"),
        median_top_fold=("top_fold", "median"),
    ).reset_index()
    grouped.insert(1, "round", round_name)
    round_rows.append(grouped)
round_qc = pd.concat(round_rows, ignore_index=True)
display(round_qc)

## Guide composition by tumor

`None` and `UNK` are excluded. Fractions are normalized within each slide/tumor combination over mapped guide calls.

In [ ]:
called_index = np.flatnonzero(mapped_call)
guide_counts = (
    pd.DataFrame({
        "slide_id": slide_values[called_index],
        "tumor_id": tumor_values[called_index],
        "guide": guide_values[called_index],
    })
    .groupby(["slide_id", "tumor_id", "guide"], observed=True, sort=False)
    .size().rename("n_cells").reset_index()
)
guide_counts["fraction_within_tumor_calls"] = (
    guide_counts["n_cells"]
    / guide_counts.groupby(["slide_id", "tumor_id"], observed=True)["n_cells"].transform("sum")
)
display(guide_counts.sort_values(["slide_id", "tumor_id", "n_cells"], ascending=[True, True, False]))

In [ ]:
matrix = guide_counts.pivot_table(
    index=["slide_id", "tumor_id"], columns="guide",
    values="fraction_within_tumor_calls", fill_value=0.0,
)
fig, ax = plt.subplots(figsize=(max(10, 0.25 * matrix.shape[1]), max(3, 0.35 * matrix.shape[0])))
image = ax.imshow(matrix.to_numpy(), aspect="auto", cmap="viridis", vmin=0)
ax.set_xticks(np.arange(matrix.shape[1]), matrix.columns, rotation=90, fontsize=7)
ax.set_yticks(np.arange(matrix.shape[0]), [f"{slide} | {tumor}" for slide, tumor in matrix.index], fontsize=7)
fig.colorbar(image, ax=ax, label="Fraction of mapped calls in tumor")
ax.set_title("Guide composition by tumor")
fig.tight_layout()
plt.show()

## Whole-slide tumor and guide maps

Visualization is reproducibly subsampled after selecting each population. Summaries above always use every cell. Colors are fixed cohort-wide so a guide has the same color on every slide.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
slide_order = pd.unique(slide_values).tolist()
tumor_categories = sorted(pd.unique(tumor_values[tumor_assigned]).tolist())
guide_categories = sorted(pd.unique(guide_values[mapped_call]).tolist())
tumor_colors = {name: plt.cm.turbo(i / max(1, len(tumor_categories) - 1)) for i, name in enumerate(tumor_categories)}
guide_colors = {name: plt.cm.turbo(i / max(1, len(guide_categories) - 1)) for i, name in enumerate(guide_categories)}

def sampled_indices(mask, maximum=MAX_POINTS_PER_SLIDE):
    indices = np.flatnonzero(mask)
    if len(indices) > maximum:
        indices = rng.choice(indices, size=maximum, replace=False)
    return indices

def scatter_categories(ax, indices, values, colors, *, size=POINT_SIZE):
    for value in pd.unique(values[indices]):
        selected = indices[values[indices] == value]
        ax.scatter(spatial[selected, 0], spatial[selected, 1], s=size, color=colors[value], linewidths=0, rasterized=True)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")

fig, axes = plt.subplots(len(slide_order), 2, figsize=(14, 6 * len(slide_order)), squeeze=False)
for row, slide_id in enumerate(slide_order):
    on_slide = slide_values == slide_id
    tumor_index = sampled_indices(on_slide & tumor_assigned)
    guide_index = sampled_indices(on_slide & mapped_call)
    scatter_categories(axes[row, 0], tumor_index, tumor_values, tumor_colors)
    scatter_categories(axes[row, 1], guide_index, guide_values, guide_colors)
    axes[row, 0].set_title(f"{slide_id}: tumor-assigned cells")
    axes[row, 1].set_title(f"{slide_id}: mapped guide calls (None/UNK hidden)")
fig.tight_layout()
plt.show()

## Selected-slide zoom

Specify the slide and global micron-coordinate bounds here. Zoom controls intentionally live next to the plot.

In [ ]:
SELECTED_SLIDE = slide_order[0]
ZOOM_UM = None  # (xmin, xmax, ymin, ymax), or None for the complete slide

on_slide = slide_values == SELECTED_SLIDE
if ZOOM_UM is not None:
    xmin, xmax, ymin, ymax = map(float, ZOOM_UM)
    in_zoom = (spatial[:, 0] >= xmin) & (spatial[:, 0] <= xmax) & (spatial[:, 1] >= ymin) & (spatial[:, 1] <= ymax)
else:
    in_zoom = np.ones(cohort.n_obs, dtype=bool)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
tumor_index = sampled_indices(on_slide & in_zoom & tumor_assigned)
guide_index = sampled_indices(on_slide & in_zoom & mapped_call)
scatter_categories(axes[0], tumor_index, tumor_values, tumor_colors, size=0.5)
scatter_categories(axes[1], guide_index, guide_values, guide_colors, size=0.5)
axes[0].set_title(f"{SELECTED_SLIDE}: tumors")
axes[1].set_title(f"{SELECTED_SLIDE}: mapped guides")
fig.tight_layout()
plt.show()

## What was persisted

The cohort `obs` contains all cell-level decoding outputs: eligibility, missing/incomplete nuclear input, each round's winner index, ratio, raw winner intensity, threshold, threshold fold, quality and pass flags, plus the decoded tuple and guide call. These support empirical cohort, slide, tumor and round QC.

Per-slide fitted threshold/scaling dictionaries are not consolidated into `cohort.uns`. They remain in each slide's `decode_settings.json` and per-slide H5AD `uns['post_analysis']`; compact `decode_funnel.csv`, `guide_counts.csv`, and `guide_counts_by_tumor.csv` files are also exported beside each slide H5AD.